In [ ]:
# Improved model with advanced feature engineering, multiple algorithms, and class imbalance handling
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                             f1_score, roc_auc_score, precision_recall_curve, auc)
import warnings
warnings.filterwarnings('ignore')

# Try to import XGBoost and LightGBM for better models
try:
    import xgboost as xgb
    has_xgboost = True
except ImportError:
    has_xgboost = False
    print("XGBoost not installed. Install with: pip install xgboost")

try:
    import lightgbm as lgb
    has_lightgbm = True
except ImportError:
    has_lightgbm = False
    print("LightGBM not installed. Install with: pip install lightgbm")

# Load and prepare data
file_path = 'data/merged_data.csv'
df = pd.read_csv(file_path, low_memory=False)

if 'calendar_start_date' in df.columns:
    df['calendar_start_date'] = pd.to_datetime(df['calendar_start_date'], errors='coerce')
    df = df.sort_values(['ISO_A0', 'calendar_start_date'])

if 'dengue_total' not in df.columns:
    raise ValueError('Expected target column dengue_total in merged_data.csv')
df['outbreak_flag'] = (df['dengue_total'] > 0).astype(int)

# IMPROVEMENT 1: Enhanced Feature Engineering
print("=" * 60)
print("IMPROVEMENT 1: Advanced Feature Engineering")
print("=" * 60)

# Create additional lag features (lag_4, lag_5, lag_6)
for i in range(4, 7):
    if f'lag_{i}' not in df.columns:
        df[f'lag_{i}'] = df.groupby('ISO_A0')['dengue_total'].shift(i)

# Create rolling statistics for temperature and precipitation
for window in [3, 7]:
    df[f'temp_rolling_mean_{window}'] = df.groupby('ISO_A0')['temperature_c'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    df[f'precip_rolling_std_{window}'] = df.groupby('ISO_A0')['precipitation_mm'].transform(
        lambda x: x.rolling(window=window, min_periods=1).std()
    )

# Create cyclical encoding for month (captures seasonal patterns)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Create cyclical encoding for day of year
df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

# Create interaction features
df['temp_precip_interaction'] = df['temperature_c'] * df['precipitation_mm']

# Select enhanced feature set
feature_cols = [
    'ISO_A0',
    'year', 'month', 'dayofyear', 'weekofyear',
    'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos',
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6',
    'temperature_c', 'precipitation_mm',
    'temp_rolling_mean_3', 'temp_rolling_mean_7',
    'precip_rolling_std_3', 'precip_rolling_std_7',
    'temp_precip_interaction'
]

model_df = df[feature_cols + ['outbreak_flag']].copy()
model_df = model_df.dropna()
print(f"Data shape after feature engineering: {model_df.shape}")
print(f"Features created: {len(feature_cols)}")
print(f"Class distribution: {model_df['outbreak_flag'].value_counts().to_dict()}")

# Encode and scale
encoder = LabelEncoder()
model_df['ISO_A0_encoded'] = encoder.fit_transform(model_df['ISO_A0'])

numeric_cols = [col for col in feature_cols if col != 'ISO_A0']
scaler = StandardScaler()
model_df[numeric_cols] = scaler.fit_transform(model_df[numeric_cols])

X = model_df[['ISO_A0_encoded'] + numeric_cols]
y = model_df['outbreak_flag']

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# IMPROVEMENT 2: Model Comparison
print("\n" + "=" * 60)
print("IMPROVEMENT 2: Training Multiple Models")
print("=" * 60)

models = {}

# Improved Random Forest with extended hyperparameter search
print("\n[1/3] Random Forest with expanded hyperparameters...")
param_grid_rf = {
    'n_estimators': [200, 200],
    'max_depth': [20, 30],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2', 0.3],
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
search_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1, verbose=0)
search_rf.fit(X_train, y_train)
models['Random Forest'] = search_rf.best_estimator_
print(f"Best RF params: {search_rf.best_params_}")
print(f"Best CV F1: {search_rf.best_score_:.4f}")

# Gradient Boosting Classifier
print("\n[2/3] Gradient Boosting Classifier...")
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train, y_train)
models['Gradient Boosting'] = gb
print("Gradient Boosting trained")

# XGBoost (if available)
if has_xgboost:
    print("\n[3/4] XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    models['XGBoost'] = xgb_model
    print("XGBoost trained")

# LightGBM (if available)
if has_lightgbm:
    print(f"\n[{3 + int(has_xgboost)}/4] LightGBM...")
    lgb_model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    models['LightGBM'] = lgb_model
    print("LightGBM trained")

# IMPROVEMENT 3: Comprehensive Model Evaluation
print("\n" + "=" * 60)
print("IMPROVEMENT 3: Model Comparison & Evaluation")
print("=" * 60)

best_model_name = None
best_f1 = 0
results_summary = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred_proba)
    
    results_summary.append({
        'Model': model_name,
        'Accuracy': acc,
        'F1-Score': f1,
        'ROC-AUC': roc
    })
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = model_name
        best_model = model

print("\n" + "-" * 60)
print(f"BEST MODEL: {best_model_name} (F1-Score: {best_f1:.4f})")
print("-" * 60)

# IMPROVEMENT 4: Detailed Evaluation of Best Model
print(f"\nDetailed Results for {best_model_name}:")
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# IMPROVEMENT 5: Threshold Optimization for better precision-recall trade-off
print("\n" + "=" * 60)
print("IMPROVEMENT 5: Threshold Optimization")
print("=" * 60)

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Default threshold: 0.5000")
print(f"Optimal F1 threshold: {optimal_threshold:.4f}")

y_pred_optimized = (y_pred_proba >= optimal_threshold).astype(int)
print(f"\nWith optimized threshold ({optimal_threshold:.4f}):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_optimized):.4f}")
print(f"  F1 Score: {f1_score(y_test, y_pred_optimized):.4f}")
print(f"  Precision: {(y_pred_optimized & y_test).sum() / y_pred_optimized.sum():.4f}")
print(f"  Recall: {(y_pred_optimized & y_test).sum() / y_test.sum():.4f}")

# Save best model and preprocessing objects
joblib.dump({
    'model': best_model,
    'label_encoder': encoder,
    'scaler': scaler,
    'feature_columns': X.columns.tolist(),
    'model_name': best_model_name,
    'optimal_threshold': optimal_threshold
}, 'random_forest_outbreak_model.joblib')
print(f"\nSaved best model ({best_model_name}) to random_forest_outbreak_model.joblib")

# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    print("\n" + "=" * 60)
    print("Top 10 Most Important Features:")
    print("=" * 60)
    importances = best_model.feature_importances_
    feature_names = X.columns.tolist()
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    for idx, row in feature_importance_df.head(10).iterrows():
        print(f"{row['feature']:30s}: {row['importance']:.4f}")


IMPROVEMENT 1: Advanced Feature Engineering
Data shape after feature engineering: (41901, 23)
Features created: 22
Class distribution: {1: 38347, 0: 3554}

IMPROVEMENT 2: Training Multiple Models

[1/3] Random Forest with expanded hyperparameters...
Best RF params: {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1: 0.9583

[2/3] Gradient Boosting Classifier...
Gradient Boosting trained

[3/4] XGBoost...
XGBoost trained

[4/4] LightGBM...
LightGBM trained

IMPROVEMENT 3: Model Comparison & Evaluation

Random Forest:
  Accuracy: 0.9232
  F1-Score: 0.9591
  ROC-AUC: 0.8470

Gradient Boosting:
  Accuracy: 0.9276
  F1-Score: 0.9617
  ROC-AUC: 0.8590

XGBoost:
  Accuracy: 0.8026
  F1-Score: 0.8825
  ROC-AUC: 0.8580

LightGBM:
  Accuracy: 0.7944
  F1-Score: 0.8767
  ROC-AUC: 0.8613

------------------------------------------------------------
BEST MODEL: Gradient Boosting (F1-Score: 0.9617)
-----------------------------------

Exception ignored in: <function ResourceTracker.__del__ at 0x7df0b3790040>
Traceback (most recent call last):
  File "/home/asif-ahammed/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/asif-ahammed/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/asif-ahammed/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x79f215388040>
Traceback (most recent call last):
  File "/home/asif-ahammed/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/asif-ahammed/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/asif-ahammed/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception igno